# 05 Extract ReEDS outputs: from 588 `outputs.h5` files to small parquet extracts

**Purpose.** This notebook is the only notebook that opens a ReEDS output file. ReEDS (the
Regional Energy Deployment System, NREL's capacity-expansion model of the US power sector) writes
one `outputs.h5` file per solved case: about 66 MB and 208 to 210 report tables. The paper uses
about 40 of those tables, and mostly only their nuclear rows. This notebook reads the 588 run files
of the eight ReEDS batches, keeps what the later notebooks read, and writes tidy parquet extracts
(`results/extracts/<batch>/<key>.parquet`) that are small enough to live in the repository. Every
later notebook (06a, 06b, 07, 08, 09, 11, 15, ...) reads those extracts with `h5io.load(batch, key)`
and never touches an h5 file. The extraction rule for each table is one row of
`data/extract_spec.csv`. When the output archive is absent, the notebook prints
"using committed extracts", checks that the committed extracts cover every batch and case of the
archive manifest, and finishes.

**Inputs.**
- The unpacked ReEDS output archive (`<batch>/<prefix><case>_outputs.h5`, 38.8 GB), found through
  the environment variable `NL_REEDS_OUTPUTS` (default: the sibling folder
  `../nuclear-learning-reeds-outputs`). Optional: without it the notebook runs in extracts mode.
- `data/reeds_outputs_manifest.csv`: the frozen list of the 588 archive files (batch, casefile,
  case, file, bytes, SHA-256).
- `data/extract_spec.csv`: the extraction rule for each (key, batch group).
- `cases_nuclearlearning_<batch>.csv` at the repository root: the case names of each batch.

**Outputs.**
- `results/extracts/<batch>/<key>.parquet`: one tidy table per (batch, key); columns `case`, the
  h5 dimension columns under their h5 names (`i`, `v`, `r`, `t`, `allh`, `sys_costs`, `*`, ...),
  and `Value`.
- `results/extracts/key_inventory.parquet`: one row per run (including the two batch-3 runs that
  never returned): which report tables the file holds, whether the mandate shadow price and the
  CO2 price exist, and a scan of every value for NaN or infinity.
- `results/extracts/manifest.csv`: one row per extract file (batch, key, cases, missing cases,
  rows, bytes, spec hash, pilot runs included).

**Paper items.** None directly. The extracts feed every item that rests on a ReEDS run: Table 1,
Fig 2a, Fig 3 to Fig 5, SF8 to SF15, SF20 to SF23, SF26, ST2 to ST4, ST10 to ST17.

**Runtime.** About 40 minutes on this machine with the archive on drive D (first run: 13 minutes
for the key inventory, which reads every value of every file, and 26 minutes for the extracts).
A re-run skips every up-to-date extract and takes under a minute; extracts mode takes about 25 seconds.

In [1]:
import sys
from pathlib import Path
_p = Path.cwd().resolve()
while not (_p / "nlpaper").is_dir():           # find z-ethan/ from wherever the kernel starts
    _p = _p.parent
sys.path.insert(0, str(_p))
import nlpaper
from nlpaper import paths, plotstyle as ps, h5io
nlpaper.setup(threads=4)

nlpaper setup: python 3.12.3 | numpy 1.26.4 | pandas 2.2.2 | matplotlib 3.8.4 | threads 4
determinism: seeds are explicit numpy Generators in each cell; results do not depend on PYTHONHASHSEED.


{'python': '3.12.3',
 'numpy': '1.26.4',
 'pandas': '2.2.2',
 'threads': 4,
 'threadpoolctl': True,
 'env_set': {'OMP_NUM_THREADS': '4',
  'MKL_NUM_THREADS': '4',
  'OPENBLAS_NUM_THREADS': '4',
  'NUMEXPR_NUM_THREADS': '4',
  'NUMBA_NUM_THREADS': '4'},
 'matplotlib': '3.8.4',
 'seed_note': None}

## 1. What an `outputs.h5` file is

ReEDS solves one linear program per model year (the sequential, or myopic, solve) and, after the
last year, writes its report parameters to `outputs.h5` (`reeds/io.py::write_output_to_h5`).
Each report parameter is one HDF5 group. Inside a group, a `columns` dataset names the columns in
order, and one dataset holds each column: the dimension columns as byte strings (technology `i`,
vintage `v`, region `r`, year `t`, timeslice `allh`, cost stream `sys_costs`, and unnamed GAMS
dimensions written as `*`, `*.1`, `*.2`), and the numbers in `Value`. The file is therefore a set
of long tables, one per parameter. ReEDS drops rows whose value is zero, and it drops a whole
group when every value is zero: a missing `nuclear_cap_price` group means that the mandate never
bound in that run, not that data were lost.

`h5io.read_key` turns one group into a DataFrame exactly as the original analysis did: byte strings
decode to `str`, `Value` stays a float, and the year columns `t` and `allt` become integers.

## 2. Batches, output prefixes and casefiles

Each batch is one ReEDS casefile at the repository root, `cases_nuclearlearning_<batch>.csv`; each
column of the casefile is one case. The runs were solved by NREL and returned as h5 files named
`<prefix><case>_outputs.h5`. The prefix is the batch name, except for three batches that were
delivered under earlier names: `smr100` runs carry `test1_`, `step4` runs `step4_`, and
`ratedesign` runs `rate_design_`. The `smr100` batch also has three pilot runs
(`smr100_eia_{lo,mid,hi}`, an earlier model vintage with a shorter year set) that are not casefile
columns; they sit in `smr100/pilots/` and enter the extracts as ordinary cases of batch `smr100`,
flagged in the manifest and the key inventory, because the step-3 checks compare against them.
Two batch-3 runs (`smr100_eo_p75_demhi`, `smr100_iaea_p75_demhi`) never returned; they are
recorded as missing. The file `batch2_bm_cop28_d9396_e2` is a byte-identical copy of
`batch3_bm_cop28_d9396_e2` (one rerun delivered under both names), kept so that every casefile
column maps to one file.

In [2]:
import hashlib
import json
import time

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 200)

MAN_ALL = pd.read_csv(paths.require(paths.DATA / "reeds_outputs_manifest.csv",
                                    "the repository (z-ethan/data)"))
# the archive manifest also lists non-ReEDS files (mc_perdraw.npz); keep the ReEDS runs
MAN = MAN_ALL[MAN_ALL["batch"].isin(h5io.BATCHES)].reset_index(drop=True)
assert len(MAN) == 588, f"the archive manifest lists the 588 run files, got {len(MAN)}"
MAN["pilot"] = MAN["case"].isin(sum(h5io.PILOTS.values(), []))

rows = []
for b in h5io.BATCHES:
    cols = h5io.cases(b)
    m = MAN[MAN["batch"] == b]
    rows.append(dict(batch=b, casefile=paths.casefile(b).name, prefix=h5io.prefix(b),
                     casefile_cases=len(cols), files=len(m), pilots=int(m["pilot"].sum()),
                     not_returned=len(set(cols) - set(m["case"])),
                     GB=round(m["bytes"].sum() / 1e9, 2)))
BATCH_MAP = pd.DataFrame(rows)
display(BATCH_MAP)
assert BATCH_MAP["files"].sum() == 588
assert BATCH_MAP["not_returned"].sum() == 2, "two batch-3 runs never returned"

,batch,casefile,prefix,casefile_cases,files,pilots,not_returned,GB
0,smr100,cases_nuclearlearning_smr100.csv,test1_,25,28,3,0,1.82
1,step4,cases_nuclearlearning_step4.csv,step4_,120,120,0,0,7.95
2,itcfb,cases_nuclearlearning_itcfb.csv,itcfb_,19,19,0,0,1.25
3,itcfbm,cases_nuclearlearning_itcfbm.csv,itcfbm_,24,24,0,0,1.58
4,ratedesign,cases_nuclearlearning_ratedesign.csv,rate_design_,66,66,0,0,4.37
5,ratedesign2,cases_nuclearlearning_ratedesign2.csv,ratedesign2_,93,93,0,0,6.13
6,batch2,cases_nuclearlearning_batch2.csv,batch2_,89,89,0,0,5.83
7,batch3,cases_nuclearlearning_batch3.csv,batch3_,151,149,0,2,9.89


## 3. What is extracted, and why each table is needed

The table below is `data/extract_spec.csv`, the only place that decides what leaves the archive.
Each row names one report parameter (`key`), the batches it applies to, a row filter
(`tech_filter`: a regular expression matched in full against the technology column `i`, or
`column=regex` for another column), an aggregation (`aggregate_to`) and the dimensions kept.
The reader census (which notebook reads which key, over which runs and dimensions) was taken by
searching every h5 read in the original analysis and its eight run-check notebooks. The rule that
follows from it: keep every dimension of a table that is small or that is filtered to the two
nuclear technologies (`nuclear`, the large light-water reactor, and `nuclear-smr`, the small
modular reactor), and aggregate only the large non-nuclear rows.

The aggregation modes used:
- blank: the table is kept as ReEDS wrote it (after the filter).
- `national_except:nuclear|nuclear-smr`: the two nuclear technologies keep every dimension; every
  other technology is summed over the region column `r` and carries `r = "national"`. A sum over
  `r` of the extract therefore still gives the national total of every technology, so the
  national-versus-regional consistency check of the run checks still works.
- `mean:*.2`: the mean of `Value` over the representative timeslices present (ReEDS drops zero
  rows, so this is the mean over the rows present, exactly as the original reader computed it).
- `dims_kept = t`: `Value` summed over every other dimension.

What each table means physically (units as ReEDS writes them: money in 2004 dollars, capacity in
MW, energy in MWh):
- `nuclear_cap_price`: the shadow price of the nuclear capacity mandate (the change in total
  system cost from one more MW of required nuclear capacity) by year. It is the support that the
  marginal mandated plant needs. ReEDS reports it on the model's annual rental basis, 2004 $/MW-yr:
  `nuclear_cap_price = nuclear_cap_price_raw / cost_scale / pvf_onm(t)`, where the raw value is the
  constraint's marginal in objective units. Multiplying by `pvf_onm` (14.7645 from 2026) turns it
  into a capitalized $/MW.
- `nuclear_cap_price_raw`: the same marginal before that conversion. `objfn_raw`: the optimal
  objective (total discounted system cost). `cost_scale`: the objective scale factor (1 here).
  `pvf_onm`, `pvf_capital`: the present-value factors that ReEDS applies to operating and capital
  costs of each solve year (in a sequential solve, `pvf_capital = 1` and `pvf_onm = 1/crf`, the
  inverse capital recovery factor). `z_rep`, `raw_inv_cost`, `raw_op_cost`: the objective of each
  solve year and its investment and operating parts. `error_check`: solver residuals.
- `systemcost`: the system cost by cost stream and year. `systemcost_techba`: the same by
  technology and region. `tax_expenditure_itc`: the investment tax credit that the government pays
  out, by year (negative = outlay).
- `cost_cap`: the overnight capital cost of each technology and year (2004 $/MW). For nuclear it is
  the learned cost path the run used. `cost_cap_fin_mult`, `cost_cap_fin_mult_noITC`: the financing
  multiplier that turns overnight cost into the financed cost the model sees, with and without the
  ITC, by technology, region and year; their ratio recovers the credit rate a run applied.
- `reduced_cost`: the LP reduced cost of each nuclear variable (how much its cost would have to
  fall before the model uses it). `heat_rate`, `cost_vom`: nuclear heat rate and variable O&M.
- `cap`, `cap_new_ann`, `ret_ann`, `gen_ann`: installed capacity, new capacity per year, retirements
  per year and annual generation by technology, region and year; `cap_nat`, `cap_new_ann_nat`,
  `gen_ann_nat`, `ret_ann_nat`: the national totals that ReEDS writes itself.
- `revenue`: market revenue of the nuclear fleet by category (energy, capacity reserve margin, RPS,
  storage charge). `valnew`: the value of the new nuclear capacity built in a year (MW, MWh and the
  energy, reserve-margin and RPS value). `reqt_price` (load rows): the energy price, here the simple
  mean over the representative timeslices by region and year.
- `repgasprice_nat`: the national natural-gas price. `co2_price`: the shadow price of the CO2 cap
  (only the batch-3 CO2-cap runs). `emit_nat`: national emissions. `invtran_out`: new transmission
  investment. These four confirm that a market-world switch acted.
- `load_rt`, `load_cat`, `load_stress`, `hours`, `dropped_load`, `curt_ann`: annual load by region,
  load by category, stress-period load (national by year), hours per timeslice, dropped load and
  curtailment. The run checks use them to confirm that the load inputs were identical across runs
  and that no load was dropped.

Tables deliberately left out: `nuclear_cap_price_ub` and `nuclear_cap_price_ub_raw` (the ceiling
dual of the equality-mandate case) exist in none of the 588 files (all zero, so ReEDS dropped
them); their absence is recorded in the key inventory. `stor_in` and `stor_out` (hourly storage
flows, about 90,000 rows each) are read by the run checks only for a no-negative-value scan; the
key inventory carries their minimum instead. The hourly generation, flow and requirement tables
are read by no analysis.

In [3]:
SPEC = h5io.read_spec()
SPEC_VIEW = SPEC.assign(spec_hash=[h5io.spec_hash(r) for _, r in SPEC.iterrows()])
display(SPEC_VIEW[["key", "batches", "tech_filter", "aggregate_to", "dims_kept", "note"]])
pairs = [(b, r["key"]) for _, r in SPEC.iterrows() for b in h5io._row_batches(r)]
assert len(pairs) == len(set(pairs)), "each (batch, key) appears in exactly one spec row"
print(f"{len(SPEC)} spec rows, {SPEC['key'].nunique()} keys, {len(pairs)} (batch, key) extracts")

,key,batches,tech_filter,aggregate_to,dims_kept,note
0,nuclear_cap_price,smr100;step4;itcfb;ratedesign;ratedesign2;batch3,,,,shadow price of the nuclear capacity mandate by year (2004$/MW-yr rental basis); absen...
1,nuclear_cap_price_raw,smr100;step4;itcfb;ratedesign;ratedesign2;batch3,,,,raw marginal of the mandate constraint in objective units (= cost_scale x pvf_onm x nu...
2,objfn_raw,all,,,,objective function value (2004$ present value)
3,cost_scale,all,,,,objective scaling factor (1 in these runs)
4,pvf_onm,all,,,,present-value factor of operating costs by year (1/crf; 14.7645 from 2026 in sequentia...
5,pvf_capital,all,,,,present-value factor of capital costs by year (1 in sequential solves)
6,z_rep,all,,,,objective value reported per solve year
7,raw_inv_cost,all,,,,investment part of the objective by year
8,raw_op_cost,all,,,,operating part of the objective by year
9,tax_expenditure_itc,all,,,,ITC tax expenditure by year (negative = government outlay)


42 spec rows, 40 keys, 309 (batch, key) extracts


## 4. Two data modes

- **Archive mode.** The unpacked archive is present (`NL_REEDS_OUTPUTS`, or the default sibling
  folder). The notebook checks each archive file against the frozen manifest (it exists and has the
  recorded byte size), builds the key inventory, and writes every extract whose spec row changed
  since the last run. An extract is up to date when its parquet file exists and the spec hash in
  `results/extracts/manifest.csv` equals the hash of its current spec row; the hash covers the
  row's filter and aggregation and the reader version, so editing one row re-extracts only that
  key.
- **Extracts mode.** The archive is absent. The notebook prints "using committed extracts" and
  only validates what is committed: every (batch, key) of the spec has a parquet file, its size
  and spec hash match the manifest, its cases belong to the batch, and the key inventory lists
  every run of the archive manifest.

In [4]:
ARCHIVE = paths.reeds_outputs()
if ARCHIVE is None:
    print("using committed extracts")
else:
    print(f"archive mode: reading ReEDS outputs from {ARCHIVE}")
    sizes = []
    for r in MAN.itertuples():
        f = ARCHIVE / r.relpath
        sizes.append(f.stat().st_size if f.exists() else -1)
    MAN["bytes_found"] = sizes
    absent = MAN[MAN["bytes_found"] < 0]
    wrong = MAN[(MAN["bytes_found"] >= 0) & (MAN["bytes_found"] != MAN["bytes"])]
    assert absent.empty, f"archive files missing: {absent['relpath'].tolist()[:5]}"
    assert wrong.empty, f"archive files with a size other than the manifest's: {wrong['relpath'].tolist()[:5]}"
    for r in MAN.itertuples():   # h5io.run_file must resolve every manifest entry
        assert h5io.run_file(r.batch, r.case) == ARCHIVE / r.relpath, r.relpath
    print(f"all {len(MAN)} archive files present with the manifest byte size "
          f"({MAN['bytes'].sum() / 1e9:.2f} GB); SHA-256 is recorded in the manifest, not re-hashed here")

archive mode: reading ReEDS outputs from D:\ReEDS files\nuclear-learning\public_archive\nuclear-learning-reeds-outputs


all 588 archive files present with the manifest byte size (38.83 GB); SHA-256 is recorded in the manifest, not re-hashed here


## 5. Key inventory and value scan

The run checks compared the full key sets of all runs in a batch, and scanned every value of every
table for NaN, infinity and impossible negatives. Those comparisons need the full files, so this
notebook does them once at extraction time and stores the result, one row per run:
`n_keys`, the sorted key list (`keys`), flags for the mandate shadow price, its ceiling variant and
the CO2 price, the number of non-finite values over all tables (`n_nonfinite`, with the tables
that hold them), and the minimum value of the tables that must not be negative (`cap`, `cap_nat`,
`load_rt`, `stor_in`, `stor_out`, `curt_ann`, `hours`). The two runs that never returned get a
row with `returned = False`. Reading every `Value` column takes about half a second per file.

In [5]:
INVENTORY_VERSION = "inventory-1"
NONNEG = ["cap", "cap_nat", "load_rt", "stor_in", "stor_out", "curt_ann", "hours"]
INV_PATH = paths.EXTRACTS / "key_inventory.parquet"
MANIFEST_PATH = paths.EXTRACTS / "manifest.csv"

def inventory_hash():
    raw = MAN[["batch", "case", "relpath", "bytes", "sha256"]].to_csv(index=False).replace(chr(13), "").encode()
    return hashlib.sha256(raw + INVENTORY_VERSION.encode() + json.dumps(NONNEG).encode()).hexdigest()[:16]

def expected_runs():
    # every casefile column plus the pilots, with the manifest row when the file exists
    out = []
    for b in h5io.BATCHES:
        m = MAN[MAN["batch"] == b].set_index("case")
        for c in h5io.cases(b, pilots=True):
            r = m.loc[c] if c in m.index else None
            out.append(dict(batch=b, case=c, pilot=c in h5io.PILOTS.get(b, []),
                            in_casefile=c in h5io.cases(b), returned=r is not None,
                            file=(r["relpath"] if r is not None else ""),
                            bytes=(int(r["bytes"]) if r is not None else 0),
                            sha256=(r["sha256"] if r is not None else "")))
    return pd.DataFrame(out)

def scan_file(path):
    import h5py
    with h5py.File(path, "r") as f:
        ks = sorted(f.keys())
        nonfin, mins = {}, {}
        for k in ks:
            v = f[k]["Value"][:]
            bad = int((~np.isfinite(v)).sum())
            if bad:
                nonfin[k] = bad
            if k in NONNEG:
                mins[k] = float(v.min()) if v.size else np.nan
    row = dict(n_keys=len(ks), keys=";".join(ks),
               has_nuclear_cap_price="nuclear_cap_price" in ks,
               has_nuclear_cap_price_ub="nuclear_cap_price_ub" in ks,
               has_co2_price="co2_price" in ks,
               n_nonfinite=int(sum(nonfin.values())),
               nonfinite_keys=";".join(f"{k}:{n}" for k, n in nonfin.items()))
    for k in NONNEG:
        row[f"min_{k}"] = mins.get(k, np.nan)
    return row

def manifest_rows():
    return pd.read_csv(MANIFEST_PATH) if MANIFEST_PATH.exists() else pd.DataFrame(
        columns=["batch", "key", "n_cases", "n_missing", "n_rows", "bytes", "spec_hash", "pilot_cases"])

T0 = time.time()
if ARCHIVE is not None:
    man_now = manifest_rows()
    prev = man_now[(man_now["batch"] == "all") & (man_now["key"] == "key_inventory")]
    fresh = INV_PATH.exists() and len(prev) == 1 and prev["spec_hash"].iloc[0] == inventory_hash()
    if fresh:
        print("skip key_inventory (up to date)")
    else:
        runs = expected_runs()
        scans = []
        for i, r in enumerate(runs.itertuples()):
            scans.append(scan_file(ARCHIVE / r.file) if r.returned else dict(n_keys=0, keys=""))
            if (i + 1) % 100 == 0:
                print(f"  scanned {i + 1}/{len(runs)} runs ({time.time() - T0:.0f} s)")
        INV = pd.concat([runs, pd.DataFrame(scans)], axis=1)
        paths.EXTRACTS.mkdir(parents=True, exist_ok=True)
        INV.to_parquet(INV_PATH, engine="pyarrow", compression="zstd", index=False)
        row = dict(batch="all", key="key_inventory", n_cases=int(INV["returned"].sum()),
                   n_missing=int((~INV["returned"]).sum()), n_rows=len(INV),
                   bytes=INV_PATH.stat().st_size, spec_hash=inventory_hash(),
                   pilot_cases=";".join(INV.loc[INV["pilot"], "case"]))
        man_now = pd.concat([man_now[~((man_now["batch"] == "all") & (man_now["key"] == "key_inventory"))],
                             pd.DataFrame([row])], ignore_index=True)
        man_now.sort_values(["batch", "key"]).to_csv(MANIFEST_PATH, index=False)
        print(f"key inventory: {len(INV)} runs scanned in {time.time() - T0:.0f} s")

skip key_inventory (up to date)


In [6]:
INV = pd.read_parquet(paths.require(INV_PATH, "notebook 05_extract_reeds_outputs (archive mode)"))
ret = INV[INV["returned"]]
print(f"{len(INV)} runs listed, {int(ret.shape[0])} returned, "
      f"{int((~INV['returned']).sum())} never returned: {INV.loc[~INV['returned'], 'case'].tolist()}")
summary = ret.groupby("batch").agg(
    runs=("case", "size"), key_counts=("n_keys", lambda s: sorted(set(s))),
    with_mandate_price=("has_nuclear_cap_price", "sum"), with_co2_price=("has_co2_price", "sum"),
    with_ceiling_price=("has_nuclear_cap_price_ub", "sum"), nonfinite_values=("n_nonfinite", "sum"))
display(summary.reindex(h5io.BATCHES))

# key-set consistency (replaces the run checks' key-set comparisons): the key sets of all runs
# may differ only by the mandate price pair and the CO2 price
OPTIONAL = {"nuclear_cap_price", "nuclear_cap_price_raw", "nuclear_cap_price_ub",
            "nuclear_cap_price_ub_raw", "co2_price"}
sets = {(r.batch, r.case): set(r.keys.split(";")) for r in ret.itertuples()}
union = set.union(*sets.values())
odd = {k: sorted((union - s) - OPTIONAL) for k, s in sets.items() if (union - s) - OPTIONAL}
print(f"{len(union)} distinct keys over all runs; {len(union - OPTIONAL)} in every run; "
      f"runs lacking other keys: {odd or 'none'}")
assert not odd, "every run holds the same report tables apart from the mandate and CO2 prices"
assert ret["has_nuclear_cap_price_ub"].sum() == 0, "the ceiling dual is all-zero (absent) in every run"
assert (ret["has_co2_price"] == (ret["case"].str.endswith("_co2cap"))).all(), \
    "co2_price exists exactly in the batch-3 CO2-cap runs"
mins = ret[[f"min_{k}" for k in NONNEG]].min()
print("minimum over all runs of tables that must not be negative:", mins.round(3).to_dict())
print("non-finite values over all tables and runs:", int(ret["n_nonfinite"].sum()))

590 runs listed, 588 returned, 2 never returned: ['smr100_iaea_p75_demhi', 'smr100_eo_p75_demhi']


,runs,key_counts,with_mandate_price,with_co2_price,with_ceiling_price,nonfinite_values
batch,,,,,,
smr100,28,"[207, 209]",27,0,0,0.0
step4,120,[209],120,0,0,0.0
itcfb,19,"[207, 209]",2,0,0,0.0
itcfbm,24,[207],0,0,0,0.0
ratedesign,66,"[207, 209]",18,0,0,0.0
ratedesign2,93,"[207, 209]",36,0,0,0.0
batch2,89,[207],0,0,0,0.0
batch3,149,"[207, 208, 209, 210]",100,54,0,0.0


210 distinct keys over all runs; 207 in every run; runs lacking other keys: none
minimum over all runs of tables that must not be negative: {'min_cap': -0.0, 'min_cap_nat': 0.008, 'min_load_rt': 726048.562, 'min_stor_in': -0.0, 'min_stor_out': -0.0, 'min_curt_ann': -0.0, 'min_hours': 0.033}
non-finite values over all tables and runs: 0


## 6. Extraction

`h5io.extract` opens each run file once per batch and reads every key that the spec wants from that
batch, applies the row's filter and aggregation, prepends the `case` column and writes one parquet
file per (batch, key) with zstd compression. A run that lacks a key (the mandate price in runs
where the mandate never bound, the CO2 price outside the CO2-cap world) is counted in `n_missing`,
not treated as an error. Up-to-date extracts are skipped.

In [7]:
if ARCHIVE is not None:
    t1 = time.time()
    before = manifest_rows()
    EXTRACT_MANIFEST = h5io.extract(SPEC, pilots=True, verbose=False)
    b_hash = {(r.batch, r.key): (r.spec_hash, r.bytes) for r in before.itertuples()}
    written = [f"{r.batch}/{r.key}" for r in EXTRACT_MANIFEST.itertuples()
               if b_hash.get((r.batch, r.key)) != (r.spec_hash, r.bytes)]
    print(f"{len(written)} extracts written, {len(pairs) - len(written)} already up to date"
          + (f": {written[:12]}{' ...' if len(written) > 12 else ''}" if written else ""))
    print(f"extraction finished in {(time.time() - t1) / 60:.1f} min "
          f"(notebook total {(time.time() - T0) / 60:.1f} min)")

0 extracts written, 309 already up to date
extraction finished in 0.0 min (notebook total 0.0 min)


## 7. Manifest and size report

The committed extracts must stay small: below 50 MB per file and 150 MB in total. The manifest
lists, for each (batch, key), the number of cases extracted, the number of runs that lack the key,
the row count, the file size and the spec hash. `pilot_cases` names the three pilot runs where
they are included (batch `smr100`).

In [8]:
M = pd.read_csv(paths.require(MANIFEST_PATH, "notebook 05_extract_reeds_outputs (archive mode)"))
files = M[M["key"] != "key_inventory"]
by_batch = files.groupby("batch").agg(extracts=("key", "size"), rows=("n_rows", "sum"),
                                      MB=("bytes", lambda s: round(s.sum() / 1e6, 2)))
display(by_batch.reindex(h5io.BATCHES))
by_key = files.groupby("key").agg(batches=("batch", "size"), rows=("n_rows", "sum"),
                                  MB=("bytes", lambda s: round(s.sum() / 1e6, 2)))
display(by_key.sort_values("MB", ascending=False).head(12))
total_mb = M["bytes"].sum() / 1e6
print(f"total committed extracts: {total_mb:.1f} MB in {len(M)} files; "
      f"largest file {M['bytes'].max() / 1e6:.2f} MB ({M.loc[M['bytes'].idxmax(), 'batch']}/"
      f"{M.loc[M['bytes'].idxmax(), 'key']})")
assert total_mb < 150, "the committed extracts stay below 150 MB in total"
assert M["bytes"].max() < 50e6, "no extract file reaches 50 MB"

,extracts,rows,MB
batch,,,
smr100,39,3531814,3.47
step4,39,5417443,10.94
itcfb,39,2470774,2.54
itcfbm,37,3093060,3.05
ratedesign,39,3164453,6.69
ratedesign2,39,4238735,7.63
batch2,37,4169735,7.84
batch3,40,6814791,13.90


,batches,rows,MB
key,,,
load_cat,8,3346453,9.87
systemcost_techba,8,3045284,9.80
revenue,8,976275,4.39
valnew,8,763240,4.17
reduced_cost,8,527204,3.20
reqt_price,8,846990,2.88
gen_ann,8,957172,2.86
cost_cap_fin_mult,8,3859651,1.93
curt_ann,8,564620,1.83


total committed extracts: 56.1 MB in 310 files; largest file 2.65 MB (batch3/load_cat)


## 8. Validation of the committed extracts (both modes)

This cell checks the extracts against the spec and the archive manifest, in either mode. Every
(batch, key) of the spec must have a manifest row whose spec hash equals the current spec row's
hash and a parquet file of the recorded size; the extracted cases plus the runs without the key
must equal the batch's runs (casefile columns plus pilots); every case in a parquet file must be a
returned run of that batch; and the key inventory must list exactly the archive manifest's runs
plus the two runs that never returned.

In [9]:
M = pd.read_csv(MANIFEST_PATH).set_index(["batch", "key"])
INV = pd.read_parquet(INV_PATH)
problems = []
returned = {b: set(g.loc[g["returned"], "case"]) for b, g in INV.groupby("batch")}
for _, row in SPEC.iterrows():
    for b in h5io._row_batches(row):
        k = row["key"]
        p = paths.EXTRACTS / b / f"{k}.parquet"
        if (b, k) not in M.index:
            problems.append(f"{b}/{k}: no manifest row")
            continue
        m = M.loc[(b, k)]
        if not p.exists():
            problems.append(f"{b}/{k}: parquet missing")
            continue
        if p.stat().st_size != m["bytes"]:
            problems.append(f"{b}/{k}: size {p.stat().st_size} != manifest {m['bytes']}")
        if m["spec_hash"] != h5io.spec_hash(row):
            problems.append(f"{b}/{k}: spec changed since extraction (re-run in archive mode)")
        if m["n_cases"] + m["n_missing"] != len(h5io.cases(b, pilots=True)):
            problems.append(f"{b}/{k}: cases {m['n_cases']} + missing {m['n_missing']} "
                            f"!= {len(h5io.cases(b, pilots=True))} runs")
        cs = set(pd.read_parquet(p, columns=["case"])["case"].unique())
        if len(cs) != m["n_cases"] or not cs <= returned[b]:
            problems.append(f"{b}/{k}: case set does not match the returned runs")
inv_runs = set(zip(INV.loc[INV["returned"], "batch"], INV.loc[INV["returned"], "case"]))
man_runs = set(zip(MAN["batch"], MAN["case"]))
if inv_runs != man_runs:
    problems.append(f"key inventory runs differ from the archive manifest: {sorted(inv_runs ^ man_runs)[:5]}")
if (~INV["returned"]).sum() != 2:
    problems.append("the key inventory should list exactly two runs that never returned")
assert not problems, "\n".join(problems)
covered = pd.DataFrame([dict(batch=b, runs_in_manifest=int((MAN["batch"] == b).sum()),
                             runs_in_extracts=len(returned.get(b, ())),
                             pilots=int(INV[(INV["batch"] == b) & INV["pilot"]].shape[0]),
                             keys=len(h5io.available(b))) for b in h5io.BATCHES])
display(covered)
print(f"extracts valid: {len(pairs)} (batch, key) files match the spec and cover all "
      f"{len(man_runs)} archive runs")

,batch,runs_in_manifest,runs_in_extracts,pilots,keys
0,smr100,28,28,3,39
1,step4,120,120,0,39
2,itcfb,19,19,0,39
3,itcfbm,24,24,0,37
4,ratedesign,66,66,0,39
5,ratedesign2,93,93,0,39
6,batch2,89,89,0,37
7,batch3,149,149,0,40


extracts valid: 309 (batch, key) files match the spec and cover all 588 archive runs


## 9. Reading an extract

Every later notebook reads an extract as below. The example is the reference case `smr100_eia_p50`:
the shadow price of the mandate in the years where it binds, converted from the model's rental basis
(2004 $/MW-yr) to 2024 $/kW-yr with the ReEDS deflator.

In [10]:
d = h5io.load("smr100", "nuclear_cap_price")
ref = d[d["case"] == "smr100_eia_p50"].set_index("t")["Value"]
print((ref / 1000 * ps.to2025(2004) / ps.to2025(2024)).round(1).rename("2024 $/kW-yr").to_string())

t
2038    405.899994
2041    337.000000
2044    325.899994
2047    292.899994
2050    310.200012
